# ClipForge — تصدير فيديو من الهاتف

افتح الدفتر في Google Colab، ارفع الفيديو واللوجو، عدّل القيم، ثم شغّل الخلايا بالترتيب. المعالجة تتم باستخدام FFmpeg خارج متصفح الهاتف.


In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg


In [ ]:
from google.colab import files
print('ارفع الفيديو')
uploaded = files.upload()
video = next(iter(uploaded))
print('ارفع لوجو PNG اختياريًا، أو اضغط اختيار بدون ملف')
logo_upload = files.upload()
logo = next(iter(logo_upload)) if logo_upload else ''


## الإعدادات
اكتب الأوقات بالثواني، وCrop كنسب مئوية.


In [ ]:
start=0.0
end=0.0
crop_x=0.0
crop_y=0.0
crop_w=100.0
crop_h=100.0
brightness=0.0
contrast=1.0
saturation=1.0
speed=1.0
text=''
text_size=44
logo_width=180
logo_x=30
logo_y=30
output='clipforge_export.mp4'


In [ ]:
import subprocess, shlex
def esc(v): return str(v).replace(chr(92),chr(92)*2).replace(':',chr(92)+':').replace(chr(39),chr(92)+chr(39))
filters=[]
if crop_w<100 or crop_h<100 or crop_x>0 or crop_y>0: filters.append(f'crop=iw*{crop_w/100}:ih*{crop_h/100}:iw*{crop_x/100}:ih*{crop_y/100}')
filters.append(f'eq=brightness={brightness}:contrast={contrast}:saturation={saturation}')
filters.append(f'setpts={1/max(speed,0.01)}*PTS')
if text: filters.append(f"drawtext=text='{esc(text)}':fontsize={text_size}:fontcolor=white:borderw=3:bordercolor=black:x=(w-text_w)/2:y=h-text_h-40")
vf=','.join(filters)
cmd=['ffmpeg','-y']
if end>start: cmd += ['-ss',str(start),'-to',str(end)]
cmd += ['-i',video]
if logo:
    cmd += ['-i',logo]
    cmd += ['-filter_complex',f'[0:v]{vf}[v0];[1:v]scale={logo_width}:-1[logo];[v0][logo]overlay={logo_x}:{logo_y}[vout]','-map','[vout]','-map','0:a?']
else:
    cmd += ['-vf',vf,'-map','0:v','-map','0:a?']
cmd += ['-c:v','libx264','-preset','veryfast','-crf','22','-c:a','aac','-movflags','+faststart',output]
print('جاري التصدير')
subprocess.run(cmd,check=True)
print('اكتمل التصدير:',output)


In [ ]:
from google.colab import files
files.download(output)
